In [1]:
import json
import os
import sys

import numpy as np
import pandas as pd

In [4]:
def geometric_adstock(x, theta, L=12, normalize=True):
    """
    Geometric (fixed-decay) adstock via a causal weighted convolution.

    The adstocked value at week t is a weighted sum of current and past spend,
    with weights theta**0, theta**1, ..., theta**L:

        a_t = sum_{l=0}^{L} theta**l * x_{t-l}   (optionally normalized)

    Implemented as a convolution rather than a recursive loop so it stays
    vectorized and differentiable - the same structure ports cleanly to
    PyTensor later.

    Parameters
    ----------
    x : array-like
        Spend series for a single channel (length T).
    theta : float in [0, 1)
        Decay/retention rate. Higher = longer carryover.
        (e.g. TV ~0.6-0.7, paid search ~0.1-0.2.)
    L : int
        Maximum lag (carryover window) in weeks.
    normalize : bool
        If True, weights sum to 1 so the adstocked series keeps the same
        scale as the input (a weighted moving average). Recommended.

    Returns
    -------
    np.ndarray of length T : the adstocked series.
    """
    x = np.asarray(x, dtype=float)
    if not (0.0 <= theta < 1.0):
        raise ValueError(f"theta must be in [0, 1), got {theta}")

    weights = theta ** np.arange(L + 1)
    if normalize:
        weights = weights / weights.sum()

    # np.convolve(x, weights)[t] = sum_l x[t-l] * weights[l], which is causal:
    # the value at t depends only on x at t and earlier. Trim to length T.
    
    # Convolution slides the weight kernel across x and, at each position, computes a sum of products. 
    
    return np.convolve(x, weights)[: len(x)]


def hill_saturation(x, alpha, kappa):
    """
    Hill (sigmoidal) saturation. Bounded in [0, 1), strictly increasing.

        s(x) = x**alpha / (kappa**alpha + x**alpha)

    kappa is the half-saturation point: s(kappa) = 0.5, which makes it
    interpretable ("the spend level at which we reach half of max effect").
    alpha controls the steepness of the S-curve.

    Parameters
    ----------
    x : array-like
        (Adstocked) spend, non-negative.
    alpha : float > 0
        Shape/steepness.
    kappa : float > 0
        Half-saturation point, on the same scale as x.

    Returns
    -------
    np.ndarray : saturated response in [0, 1).
    """
    x = np.asarray(x, dtype=float)
    if alpha <= 0 or kappa <= 0:
        raise ValueError("alpha and kappa must be positive")
    # guard against 0**negative etc.; x is expected non-negative
    xa = np.power(np.clip(x, 0, None), alpha)
    return xa / (kappa ** alpha + xa)


def logistic_saturation(x, lam):
    """
    Logistic saturation - the alternative form used in the functional-form
    sensitivity analysis (swap this in for hill_saturation and watch ROI move).

        s(x) = (1 - exp(-lam * x)) / (1 + exp(-lam * x))

    Parameters
    ----------
    x : array-like
        (Adstocked) spend, non-negative.
    lam : float > 0
        Saturation rate.

    Returns
    -------
    np.ndarray : saturated response in [0, 1).
    """
    x = np.asarray(x, dtype=float)
    if lam <= 0:
        raise ValueError("lam must be positive")
    return (1 - np.exp(-lam * x)) / (1 + np.exp(-lam * x))


In [5]:
SEED = 42
N_WEEKS = 156          # 3 years of weekly data
ADSTOCK_L = 12         # carryover window (weeks)
START_DATE = "2021-01-04"

In [6]:
CHANNELS = {
    "tv":      dict(decay=0.65, alpha=2.0, kappa=28.0, target_roi=3.0, spend_mean=30.0, spend_sd=7.0, demand_beta=0.45),
    "search":  dict(decay=0.15, alpha=1.6, kappa=18.0, target_roi=4.5, spend_mean=20.0, spend_sd=5.0, demand_beta=0.60),
    "social":  dict(decay=0.35, alpha=1.8, kappa=16.0, target_roi=2.5, spend_mean=18.0, spend_sd=4.5, demand_beta=0.30),
    "display": dict(decay=0.45, alpha=1.5, kappa=11.0, target_roi=1.8, spend_mean=12.0, spend_sd=3.0, demand_beta=0.20),
    "email":   dict(decay=0.20, alpha=1.7, kappa=5.0,  target_roi=3.5, spend_mean=6.0,  spend_sd=1.5, demand_beta=0.10),
}

# Baseline (non-media) structure - thousands of dollars.
INTERCEPT = 300.0             # baseline weekly revenue with zero media
TREND_PER_WEEK = 0.6          # gentle upward drift
SEASON_AMPLITUDE = 40.0       # yearly seasonality strength
N_FOURIER = 3                 # number of Fourier pairs for seasonality
DEMAND_ON_SALES = 45.0        # how much the hidden demand driver moves revenue
NOISE_SD = 18.0 

In [7]:
rng = np.random.default_rng(SEED)

In [9]:
t = np.arange(N_WEEKS)

In [10]:
t

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155])

In [11]:
trend = TREND_PER_WEEK * t

In [12]:
trend

array([ 0. ,  0.6,  1.2,  1.8,  2.4,  3. ,  3.6,  4.2,  4.8,  5.4,  6. ,
        6.6,  7.2,  7.8,  8.4,  9. ,  9.6, 10.2, 10.8, 11.4, 12. , 12.6,
       13.2, 13.8, 14.4, 15. , 15.6, 16.2, 16.8, 17.4, 18. , 18.6, 19.2,
       19.8, 20.4, 21. , 21.6, 22.2, 22.8, 23.4, 24. , 24.6, 25.2, 25.8,
       26.4, 27. , 27.6, 28.2, 28.8, 29.4, 30. , 30.6, 31.2, 31.8, 32.4,
       33. , 33.6, 34.2, 34.8, 35.4, 36. , 36.6, 37.2, 37.8, 38.4, 39. ,
       39.6, 40.2, 40.8, 41.4, 42. , 42.6, 43.2, 43.8, 44.4, 45. , 45.6,
       46.2, 46.8, 47.4, 48. , 48.6, 49.2, 49.8, 50.4, 51. , 51.6, 52.2,
       52.8, 53.4, 54. , 54.6, 55.2, 55.8, 56.4, 57. , 57.6, 58.2, 58.8,
       59.4, 60. , 60.6, 61.2, 61.8, 62.4, 63. , 63.6, 64.2, 64.8, 65.4,
       66. , 66.6, 67.2, 67.8, 68.4, 69. , 69.6, 70.2, 70.8, 71.4, 72. ,
       72.6, 73.2, 73.8, 74.4, 75. , 75.6, 76.2, 76.8, 77.4, 78. , 78.6,
       79.2, 79.8, 80.4, 81. , 81.6, 82.2, 82.8, 83.4, 84. , 84.6, 85.2,
       85.8, 86.4, 87. , 87.6, 88.2, 88.8, 89.4, 90

In [ ]:
def fourier_seasonality(t, period=52.13, n_terms=N_FOURIER, rng=None):
    """Smooth yearly seasonality as a sum of sine/cosine pairs."""
    # period is number of weeks in a year
    # n_terms=3 is how many harmonics to sum
    season = np.zeros_like(t, dtype=float)
    coeffs = []
    for k in range(1, n_terms + 1):
        a = rng.normal(0, 1)
        b = rng.normal(0, 1)
        coeffs.append((a, b))
        season += a * np.sin(2 * np.pi * k * t / period) + b * np.cos(2 * np.pi * k * t / period)
    # normalize to unit std then scale, so amplitude is controllable
    season = season / season.std()
    return season, coeffs

In [17]:
season = np.zeros_like(t, dtype=float)

In [18]:
season

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

In [19]:
coeffs = []

In [ ]:
   for k in range(1, 3 + 1):
        # print(k)
        a = rng.normal(0, 1)

        # b = rng.normal(0, 1)
        break
        # coeffs.append((a, b))
        # season += a * np.sin(2 * np.pi * k * t / period) + b * np.cos(2 * np.pi * k * t / period)

1
2
3


In [23]:
a = rng.normal(0, 1)

In [28]:
b = rng.normal(0, 1)

In [29]:
period=52.13

In [30]:
season += a * np.sin(2 * np.pi * k * t / period) + b * np.cos(2 * np.pi * k * t / period)

In [31]:
season

array([-0.01680116, -0.12758856, -0.2218753 , -0.28746753, -0.31588239,
       -0.30344508, -0.25176408, -0.16752313, -0.0616169 ,  0.05225808,
        0.15937467,  0.24587978,  0.30058595,  0.31641819,  0.29132896,
        0.22856298,  0.1362376 ,  0.02629299, -0.08705202, -0.18913883,
       -0.26676486, -0.30989094, -0.3129397 , -0.27551685, -0.20246218,
       -0.10322366,  0.00936449,  0.12074156,  0.21650346,  0.28426558,
        0.31526444,  0.30549104,  0.25620934,  0.17379281,  0.06890015,
       -0.04490317, -0.15289929, -0.24112138, -0.29815992, -0.31663828,
       -0.29416671, -0.23365139, -0.14291859, -0.03370254,  0.07987217,
        0.18311724,  0.26268027,  0.3082716 ,  0.31399504,  0.27911039,
        0.20812917,  0.1102312 , -0.00192266, -0.11382786, -0.21101204,
       -0.28090662, -0.31447234, -0.30736824, -0.26051308, -0.17996649,
       -0.07614535,  0.03752345,  0.14633946,  0.23622979,  0.2955692 ,
        0.31668347,  0.29684197,  0.23861073,  0.14952064,  0.04

In [14]:
season_raw, season_coeffs = fourier_seasonality(t, rng=rng)

In [15]:
season_raw

array([-0.6947686 , -0.89793026, -1.00291978, -0.98967557, -0.85507409,
       -0.61343537, -0.29464061,  0.05989005,  0.40282671,  0.68741843,
        0.87397681,  0.93548644,  0.86158437,  0.6603614 ,  0.35772073,
       -0.00565104, -0.37933197, -0.70959584, -0.94640387, -1.0500542 ,
       -0.99662259, -0.78147597, -0.42038315,  0.05194854,  0.58575266,
        1.12203168,  1.59961429,  1.96256222,  2.16699843,  2.18649792,
        2.01536218,  1.66936378,  1.18386732,  0.60956313,  0.0063513 ,
       -0.56385471, -1.04450195, -1.391028  , -1.5759247 , -1.59161659,
       -1.45081636, -1.1843428 , -0.83671004, -0.46007855, -0.10736313,
        0.17460655,  0.35099189,  0.40316601,  0.33072243,  0.15103758,
       -0.10352632, -0.39038308, -0.66279924, -0.87646672, -0.99569111,
       -0.99837003, -0.87909791, -0.64998389, -0.33907229,  0.01342599,
        0.36059237,  0.655225  ,  0.85637327,  0.93510796,  0.87874911,
        0.69297193,  0.40148684,  0.04330641, -0.33207122, -0.67

In [16]:
season_coeffs

[(0.30471707975443135, -1.0399841062404955),
 (0.7504511958064572, 0.9405647163912139),
 (-1.9510351886538364, -1.302179506862318)]

In [33]:
base = rng.normal(20, 5, size=N_WEEKS)

In [34]:
base

array([15.73478036, 24.39698987, 23.88895968, 20.33015349, 25.63620603,
       22.33754671, 15.70353769, 21.84375392, 15.205587  , 24.39225151,
       19.75037045, 19.07568818, 16.59535228, 26.11270669, 19.22735259,
       17.85836089, 18.23933225, 22.66154593, 21.82722032, 22.06366306,
       22.15410502, 30.708238  , 17.96792492, 17.43878635, 15.93113636,
       23.07989711, 25.64486146, 19.43026271, 15.79921762, 15.87759392,
       23.25296394, 23.71627086, 22.71577134, 16.67245146, 21.16080662,
       20.58342905, 21.09344298, 24.35714389, 21.11797774, 23.39456782,
       20.33789535, 21.44559699, 23.15644113, 12.7142209 , 18.40164392,
       17.64813673, 16.80561076, 18.62428874, 27.47470656, 15.67084442,
       24.84139177, 11.58565114, 18.32557485, 20.81376533, 22.93111166,
       23.5561329 , 23.96673618, 18.25637464, 17.68824104, 24.28987941,
       19.04347838, 13.62156838, 14.33356393, 15.40273857, 22.48580372,
       20.71212868, 23.45242677, 17.86373677, 20.79269846, 23.12